# Βήμα 4 – Σύστημα Α: τελική απόφαση και σύγκριση με κλασικά μοντέλα
1. Κάνουμε τις τρεις εξόδους του δικτύου **συνεπείς μεταξύ τους**.
2. Συγκρίνουμε το νευρωνικό με **Logistic Regression** και **Random Forest** στα ίδια δεδομένα.
3. Βλέπουμε πώς αλλάζουν οι ψευδείς συναγερμοί με το **κατώφλι απόφασης**.

Δεν γίνεται νέα εκπαίδευση του δικτύου· φορτώνουμε το αποθηκευμένο μοντέλο.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"
SEED = 42
warnings.filterwarnings("ignore", category=ConvergenceWarning)

d = np.load(DATA + "unsw_prepared.npz")
with open(MODELS + "unsw_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
ATTACK_CLASSES, SEVERITY_CLASSES = meta["attack_classes"], meta["severity_classes"]
SEV_WEIGHTS = np.array([meta["severity_weight"][c] for c in SEVERITY_CLASSES])
# σοβαρότητα κάθε τύπου επίθεσης, ως αριθμός κλάσης (π.χ. Exploits -> High -> 3)
SEV_OF_ATTACK = np.array([SEVERITY_CLASSES.index(meta["severity_map"][c]) for c in ATTACK_CLASSES])

X_train, X_val, X_test = d["X_train"], d["X_val"], d["X_test"]
y = {s: {k: d[f"y_{s}_{k}"].astype(int) for k in ("label", "attack_cat", "severity")} for s in ("train", "val", "test")}

# compile=False: φορτώνουμε το δίκτυο μόνο για προβλέψεις (δεν χρειάζεται η συνάρτηση σφάλματος)
model = keras.models.load_model(MODELS + "unsw_multitask.keras", compile=False)

def predict(X):
    p_label, p_attack, p_sev = model.predict(X, batch_size=1024, verbose=0)
    return p_label.ravel(), p_attack, p_sev

p_label, p_attack, p_sev = predict(X_test)
pv_label, _, _ = predict(X_val)
print("OK:", p_label.shape, p_attack.shape, p_sev.shape)

## 4.1 Συνεπής τελική απόφαση
Οι τρεις έξοδοι αποφασίζουν ανεξάρτητα, οπότε μπορεί να διαφωνούν (π.χ. «Normal» αλλά «σοβαρότητα Low»). Ορίζουμε ιεραρχικό κανόνα:
1. Η 1η έξοδος αποφασίζει αν είναι επίθεση: `P(attack) ≥ κατώφλι`.
2. Αν **όχι** → τύπος = Normal, σοβαρότητα = None.
3. Αν **ναι** → τύπος = ο πιο πιθανός **τύπος επίθεσης** (χωρίς το Normal) και σοβαρότητα = αυτή που αντιστοιχεί στον τύπο.

In [ ]:
def decide(p_label, p_attack, threshold=0.5):
    is_attack = p_label >= threshold
    attack = np.where(is_attack, p_attack[:, 1:].argmax(1) + 1, 0)     # 0 = Normal
    severity = SEV_OF_ATTACK[attack]
    return is_attack.astype(int), attack, severity

def risk_score(p_label, p_sev):
    p = p_sev[:, 1:]
    return (100 * p_label * (p * SEV_WEIGHTS[1:]).sum(1) / np.clip(p.sum(1), 1e-9, None)).astype("float64")

yt = y["test"]
raw = {"label": (p_label >= 0.5).astype(int), "attack_cat": p_attack.argmax(1), "severity": p_sev.argmax(1)}
lab, att, sev = decide(p_label, p_attack)
cons = {"label": lab, "attack_cat": att, "severity": sev}

def summary(pred):
    return {"label acc": accuracy_score(yt["label"], pred["label"]),
            "attack_cat acc": accuracy_score(yt["attack_cat"], pred["attack_cat"]),
            "attack_cat macro-F1": f1_score(yt["attack_cat"], pred["attack_cat"], average="macro"),
            "severity acc": accuracy_score(yt["severity"], pred["severity"]),
            "severity macro-F1": f1_score(yt["severity"], pred["severity"], average="macro")}

cmp_decision = pd.DataFrame({"Ανεξάρτητες έξοδοι": summary(raw), "Συνεπής απόφαση": summary(cons)}).round(4)
cmp_decision

In [ ]:
print(classification_report(yt["attack_cat"], cons["attack_cat"], labels=range(len(ATTACK_CLASSES)),
                            target_names=ATTACK_CLASSES, digits=3, zero_division=0))

def plot_cm(y_true, y_pred, names, title, fname, size=(5, 4.2)):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(names)))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    annot = np.array([[f"{cmn[i, j]:.0%}\n{cm[i, j]}" for j in range(len(names))] for i in range(len(names))])
    fig, ax = plt.subplots(figsize=size)
    sns.heatmap(cmn, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=names, yticklabels=names, annot_kws={"size": 8}, ax=ax)
    ax.set_xlabel("Πρόβλεψη"); ax.set_ylabel("Πραγματικό"); ax.set_title(title)
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout(); plt.savefig(FIG + fname, dpi=200); plt.show()

plot_cm(yt["attack_cat"], cons["attack_cat"], ATTACK_CLASSES, "Τύπος επίθεσης – συνεπής απόφαση (test)",
        "19_unsw_cm_attack_cat_consistent.png", size=(9.5, 8))
plot_cm(yt["severity"], cons["severity"], SEVERITY_CLASSES, "Σοβαρότητα – συνεπής απόφαση (test)",
        "20_unsw_cm_severity_consistent.png", size=(5.5, 4.6))

## 4.2 Σύγκριση με κλασικά μοντέλα
Ίδια δεδομένα εκπαίδευσης, ίδιο test set. Κάθε κλασικό μοντέλο εκπαιδεύεται δύο φορές: μία για το `label` και μία για το `attack_cat`.
(Το Random Forest θέλει 1–3 λεπτά.)

In [ ]:
def make_lr():
    return LogisticRegression(max_iter=500, class_weight="balanced")

def make_rf():
    return RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)

rows = {}
for name, make in [("Logistic Regression", make_lr), ("Random Forest", make_rf)]:
    m_label = make().fit(X_train, y["train"]["label"])
    m_attack = make().fit(X_train, y["train"]["attack_cat"])
    pl = m_label.predict_proba(X_test)[:, 1]
    pa = m_attack.predict(X_test)
    rows[name] = {"label acc": accuracy_score(yt["label"], pl >= 0.5),
                  "label precision": precision_score(yt["label"], pl >= 0.5),
                  "label recall": recall_score(yt["label"], pl >= 0.5),
                  "label F1": f1_score(yt["label"], pl >= 0.5),
                  "label ROC-AUC": roc_auc_score(yt["label"], pl),
                  "attack_cat acc": accuracy_score(yt["attack_cat"], pa),
                  "attack_cat macro-F1": f1_score(yt["attack_cat"], pa, average="macro")}
    print(name, "έτοιμο")

rows["Νευρωνικό (multi-task)"] = {
    "label acc": accuracy_score(yt["label"], cons["label"]),
    "label precision": precision_score(yt["label"], cons["label"]),
    "label recall": recall_score(yt["label"], cons["label"]),
    "label F1": f1_score(yt["label"], cons["label"]),
    "label ROC-AUC": roc_auc_score(yt["label"], p_label),
    "attack_cat acc": accuracy_score(yt["attack_cat"], cons["attack_cat"]),
    "attack_cat macro-F1": f1_score(yt["attack_cat"], cons["attack_cat"], average="macro")}

comparison = pd.DataFrame(rows).T.round(4)
comparison.to_csv(MODELS + "unsw_model_comparison.csv")
comparison

## 4.3 Κατώφλι απόφασης και ψευδείς συναγερμοί
Το κατώφλι 0.5 δεν είναι υποχρεωτικό. Ανεβάζοντάς το, το σύστημα γίνεται λιγότερο «καχύποπτο»: λιγότεροι ψευδείς συναγερμοί, αλλά χάνει περισσότερες επιθέσεις.
Προσοχή: το κατώφλι πρέπει να **επιλέγεται στο validation**, ποτέ στο test. Ο πίνακας δείχνει και τα δύο, για να φανεί αν το validation «συμφωνεί» με το test.

In [ ]:
def at_threshold(y_true, p, t):
    pred = p >= t
    return {"accuracy": accuracy_score(y_true, pred), "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred),
            "FPR": ((pred == 1) & (y_true == 0)).sum() / (y_true == 0).sum()}

ths = np.round(np.arange(0.05, 1.0, 0.05), 2)
curve_test = pd.DataFrame([at_threshold(yt["label"], p_label, t) for t in ths], index=ths)
curve_val = pd.DataFrame([at_threshold(y["val"]["label"], pv_label, t) for t in ths], index=ths)
print("Καλύτερο κατώφλι για accuracy στο validation:", curve_val["accuracy"].idxmax(),
      "| στο test:", curve_test["accuracy"].idxmax())

thr_table = pd.concat({"validation": curve_val, "test": curve_test}, axis=1).loc[[0.3, 0.5, 0.7, 0.9]].round(4)
thr_table.index.name = "κατώφλι"
print(thr_table.to_string())

fpr, tpr, _ = roc_curve(yt["label"], p_label)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(fpr, tpr, color="#2a78d6", lw=2, label=f"Νευρωνικό (AUC = {roc_auc_score(yt['label'], p_label):.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", ls="--", lw=1, label="Τυχαία πρόβλεψη")
axes[0].set_xlabel("Ψευδείς συναγερμοί (FPR)"); axes[0].set_ylabel("Recall (TPR)"); axes[0].set_title("Καμπύλη ROC (test)"); axes[0].legend()
axes[1].plot(ths, curve_test["recall"], color="#2a78d6", lw=2, marker="o", ms=4, label="Recall")
axes[1].plot(ths, curve_test["precision"], color="#eb6834", lw=2, marker="o", ms=4, label="Precision")
axes[1].plot(ths, curve_test["FPR"], color="#1baf7a", lw=2, marker="o", ms=4, label="Ψευδείς συναγερμοί (FPR)")
axes[1].axvline(0.5, color="gray", ls="--", lw=1)
axes[1].set_xlabel("Κατώφλι απόφασης"); axes[1].set_title("Επίδραση του κατωφλιού (test)"); axes[1].legend()
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG + "21_unsw_roc_threshold.png", dpi=200); plt.show()